### Case Study 2: Representing clinical text as vectors

Two ways to represent text as vectors are TF-IDF, and word embeddings. Here, I will walk through implementing TF-IDF on MIMIC-III discharge summaries and begin exploring simple embeddings from PubMed. 

The data is a subset of the MIMIC-III discharge summaries. This requires some credentials before data access.

In [1]:
import pandas as pd
import nltk

from math import log
from collections import defaultdict
import random

In [2]:
discharge_summaries = pd.read_csv('data.csv')
print(discharge_summaries)

       HADM_ID                                               TEXT
0     121936.0  Admission Date:  [**2125-2-9**]              D...
1     155131.0  Admission Date:  [**2131-12-23**]             ...
2     113012.0  Admission Date:  [**2194-1-7**]              D...
3     125055.0  Admission Date:  [**2189-3-17**]       Dischar...
4     139542.0  Admission Date:  [**2146-7-1**]              D...
...        ...                                                ...
3663  132211.0  Name:  [**Known lastname 17303**], [**Known fi...
3664  190484.0  Name:  [**Known lastname 400**],[**Known first...
3665  109285.0  Name:  [**Known lastname 18278**],[**Known fir...
3666  183977.0  Name:  [**Known lastname 18438**],[**Known fir...
3667  155984.0  Name:  [**Known lastname **],[**Known firstnam...

[3668 rows x 2 columns]


From the 3668 discharge summaries from the MIMIC-III dataset, I'll first define the corpus and convert to lowercase. Text normalization like this is important so that words like "Sepsis" and "sepsis" can be counted as one.

In [3]:
corpus = discharge_summaries.TEXT.str.lower()
print(corpus[0])

admission date:  [**2125-2-9**]              discharge date:   [**2125-2-16**]


service: medicine

allergies:
zocor / lescol

attending:[**doctor last name 1857**]
chief complaint:
chest pain

major surgical or invasive procedure:
central venous line insertion (right internal jugular vein)

history of present illness:
mr. [**known lastname 1858**] is an 84 yo man with moderate aortic stenosis (outside
hospital echo in [**2124**] with [**location (un) 109**] 1 cm2, gradient 28 mmhg, moderate
mitral regurgitation, mild aortic insufficiency), chronic left
ventricular systolic heart failure with ef 25-30%, hypertension,
hyperlipidemia, diabetes mellitus, cad s/p cabg in [**2099**] with
svg-lad-diagonal, svg-om, and svg-rpda-rpl, with a re-do cabg in
[**9-/2117**] with lima-lad, svg-om, svg-diagonal, and svg-rca. he also
has severe peripheral arterial disease s/p peripheral bypass
surgery. he presented to [**hospital 1474**] hospital er this morning with
shortness of breath and chest pain 

#### TF-IDF
To implement TF-IDF, I will need to define a $1 \times N$ vocabulary vector. Let's take a look at the distribution of terms across our corpus.

In [4]:
def count_words(corpus: list) -> list:
    word_counts = defaultdict(int)
    for document in corpus:
        words = nltk.word_tokenize(document)
        for word in words:
            word_counts[word] += 1

    return word_counts

word_counts = count_words(corpus)

In [5]:
# do this for speed
tokenized_corpus = [nltk.word_tokenize(document) for document in corpus]

#### Q1a. How many unique words are in our corpus?

In [6]:
len(word_counts)

122605

#### Q1b. How many unique words occur exactly once in our corpus?

In [7]:
one_unique_words = 0
for word, count in word_counts.items():
    if count  == 1:
        one_unique_words += 1
print(one_unique_words)

61248


#### Q1c. How many unique words in our corpus have a count > 5?

In [8]:
greater_than_five_unique_words = 0
for word, count in word_counts.items():
    if count  > 5:
        greater_than_five_unique_words += 1
print(greater_than_five_unique_words)

27668


#### Q2
As seen above, many words occur only once in our corpus. As discussed in class, when creating a vector of word counts this results in a very sparse representation. We can mitigate this somewhat with preprocessing, lemmatization, better tokenization, and by only preserving common words. Here, let's define our vocabulary as any word with a count > 5.

In [9]:
MIN_COUNT = 5
def create_vocabulary(corpus: list) -> list:
    vocabulary = []
    word_counts = count_words(corpus)
    for word, count in word_counts.items():
        if count > MIN_COUNT:
            vocabulary.append(word)
    return vocabulary

vocabulary = create_vocabulary(corpus)

#### Q3 Create a TF vector for each document

Since the term frequency is the number of times the term appeared in one document, and recall, there are $3,668$ documents.

In [10]:
# speed
from collections import Counter 

def calculate_tf(corpus: list, vocabulary: list) -> list:
    tfs = []
    for words in corpus:
        # get a document level vector since it'll vary per document
        doc_vec = []
        # helps you count faster
        counter = Counter(words)
        for vocab in vocabulary:
            doc_vec.append(counter.get(vocab, 0))
        tfs.append(doc_vec)
    return tfs
        
tfs = calculate_tf(tokenized_corpus, vocabulary)

In [ ]:
print(len(tfs), len(tfs[0]))

3668 27668


# Q4 - Complete the code below
Let's now create the IDF vector

In [12]:
def calculate_idf(corpus: list, vocabulary: list) -> list:
    pass

idf = calculate_idf(corpus, vocabulary)

# Q5 - Complete the code below
Let's combine our TF and IDF implementations to calculate the TF-IDF for each discharge summary

In [13]:
def calculate_tf_idf(tf: list, idf: list) -> list:
    pass

tf_idfs = calculate_tf_idf(tfs, idf)

Let's take a look at the TF-IDF representation of one of our discharge summaries

In [14]:
print(tf_idfs[random.randint(0, len(corpus))])

TypeError: 'NoneType' object is not subscriptable

We now have a $3668 \times 27668$ matrix representation of our corpus. One simple use for this is search: given one patient's discharge summary can we find a similar one? To do this we can use the cosine similarity between any two vectors in our corpus.

In [ ]:
import numpy as np
def calculate_cos_similarity(idx, tf_idfs):
    vector = tf_idfs[idx]
    cos_similarities = []
    for i, v in enumerate(tf_idfs):
        if i != idx:
            cos = (
                np.dot(vector, v)
                /
                (np.linalg.norm(vector)*np.linalg.norm(v))
            )

            cos_similarities.append(cos)
        else:
            cos_similarities.append(0)

    return cos_similarities

cos_similarities = calculate_cos_similarity(1000, tf_idfs)

In [ ]:
print(corpus[1000])

In [ ]:
print(corpus[np.argmax(cos_similarities)])

# Q6 - Answer the question below
What similarities, if any, do you see between the two documents?

# Word Embeddings
In class, we discussed a few approaches to learning word embeddings. Here, we'll explore some pretrained embeddings. First, download the pubmed word embeddings here: https://drive.google.com/file/d/1178sxpilwI7tQxSirD3r0xS_UF0A0DoG/view?usp=sharing 

Unzip the file and load it into memory:

In [ ]:
from gensim.models.keyedvectors import KeyedVectors
word_vectors_pubmed = KeyedVectors.load_word2vec_format('pubmed2018_w2v_200D.bin',binary=True, limit=200000)

Here are what the embeddings look like:

In [ ]:
print("Word:", 'sepsis')
print("Embedding:", word_vectors_pubmed['sepsis'])

We saw the canonical example of "king - man + woman = queen"; let's reproduce this in the medical domain using these pubmed embeddings.

We can use the "most_similar" method here (it implements cosine similarity) with the "positive" keyword for vector addition and the "negative" keyword for vector subtraction.

In [ ]:
word_vectors_pubmed.most_similar(positive=['severe', 'infection'])

# Q7 - What do you observe?

# Q8 - Experiment
Try some other vector arithmetic of interest and report your results below